In [1]:
!pip install huggingface_hub


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import InferenceClient
import os

In [3]:
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"]
)

In [4]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain what an LLM is in one simple sentence."
        }
    ]
)

print(response.choices[0].message.content)

An LLM is a large language model—a type of AI that learns patterns in text to understand and generate human language.


In [5]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": """
You are an AI meeting scheduling assistant.

Extract the following information from the user's meeting request:
- participant
- date
- time
- duration

Return ONLY valid JSON.
Do not add explanations, markdown, or extra text.

Example:
{
    "participant": "Ali",
    "date": "tomorrow",
    "time": "3:00 PM",
    "duration": "1 hour"
}
"""
        },
        {
            "role": "user",
            "content": "I need a 1-hour meeting with Ali tomorrow at 3 PM."
        }
    ]
)

print(response.choices[0].message.content)

{
    "participant": "Ali",
    "date": "tomorrow",
    "time": "3 PM",
    "duration": "1 hour"
}


In [7]:
import os

print(os.path.exists(r"D:\AI_Meeting_Scheduler_Assistant"))

True


In [8]:
import os

print(os.listdir(r"D:\AI_Meeting_Scheduler_Assistant"))

['2026-08-09 04-57-31.mp4', 'Ai_Meeting_Scheduler_Assistant.zip', 'data', 'notebook', 'README.md.txt', 'requirements.txt.txt', 'Week_2']


In [9]:
import os

print(os.listdir(r"D:\AI_Meeting_Scheduler_Assistant\data"))

['mock_calendar.csv']


In [10]:
import pandas as pd

df = pd.read_csv(r"D:\AI_Meeting_Scheduler_Assistant\data\mock_calendar.csv")

df.head()

,day,time,status
0,Monday,09:00,available
1,Monday,10:00,available
2,Monday,14:00,available
3,Monday,15:00,available
4,Tuesday,11:00,available


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   day     15 non-null     str  
 1   time    15 non-null     str  
 2   status  15 non-null     str  
dtypes: str(3)
memory usage: 492.0 bytes


In [12]:
df

,day,time,status
0,Monday,09:00,available
1,Monday,10:00,available
2,Monday,14:00,available
3,Monday,15:00,available
4,Tuesday,11:00,available
5,Tuesday,13:00,available
6,Tuesday,16:00,available
7,Wednesday,09:00,available
8,Wednesday,14:00,available
9,Wednesday,17:00,available


In [14]:
find_available_slots("Monday")

,day,time
0,Monday,09:00
1,Monday,10:00
2,Monday,14:00
3,Monday,15:00


In [15]:
def find_available_slots(day=None, time=None):
    available_slots = df[df["status"] == "available"]

    if day:
        available_slots = available_slots[
            available_slots["day"].str.lower() == day.lower()
        ]

    if time:
        available_slots = available_slots[
            available_slots["time"] == time
        ]

    return available_slots[["day", "time"]]

In [16]:
find_available_slots("Monday", "14:00")

,day,time
2,Monday,14:00


In [17]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    provider="auto",
    api_key=os.getenv("HF_TOKEN")
)

In [18]:
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What does the user mean by: 'Can we have a meeting on Monday at 2 PM?'"
        }
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

The user is asking if it is possible to schedule a meeting on Monday at 2 PM. They are inquiring about the availability of the people involved and whether the time slot is suitable for everyone to attend the meeting.


In [19]:
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {
            "role": "system",
            "content": """You are a meeting scheduling assistant.

Extract the meeting day and time from the user's request.

Rules:
- Return the day exactly as Monday, Tuesday, Wednesday, Thursday, or Friday.
- Convert all times to 24-hour HH:MM format.
- For example, 2 PM must become 14:00.
- 10 AM must become 10:00.
- 3 PM must become 15:00.

Return ONLY:
day: <day>
time: <HH:MM>

If no specific day is given:
day: None

If no specific time is given:
time: None"""
        },
        {
            "role": "user",
            "content": "Can we have a meeting on Monday at 2 PM?"
        }
    ],
    max_tokens=50
)

print(response.choices[0].message.content)

day: Monday
time: 14:00


In [20]:
def schedule_meeting(user_request):
    # Ask the LLM to extract day and time
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[
            {
                "role": "system",
                "content": """You are a meeting scheduling assistant.

Extract the meeting day and time from the user's request.

Rules:
- Return the day exactly as Monday, Tuesday, Wednesday, Thursday, or Friday.
- Convert all times to 24-hour HH:MM format.
- For example, 2 PM must become 14:00.
- 10 AM must become 10:00.
- 3 PM must become 15:00.

Return ONLY:
day: <day>
time: <HH:MM>

If no specific day is given:
day: None

If no specific time is given:
time: None"""
            },
            {
                "role": "user",
                "content": user_request
            }
        ],
        max_tokens=50
    )

    llm_output = response.choices[0].message.content
    print("LLM output:")
    print(llm_output)

    # Extract day and time from LLM response
    lines = llm_output.strip().split("\n")

    day = lines[0].split(":", 1)[1].strip()
    time = lines[1].split(":", 1)[1].strip()

    # Check the calendar
    matching_slots = df[
        (df["day"].str.lower() == day.lower()) &
        (df["time"] == time) &
        (df["status"] == "available")
    ]

    # Return result
    if len(matching_slots) > 0:
        return f"Yes, {day} at {time} is available."
    else:
        return f"Sorry, {day} at {time} is not available."

In [21]:
result = schedule_meeting("Can we have a meeting on Monday at 2 PM?")

print(result)

LLM output:
day: Monday
time: 14:00
Yes, Monday at 14:00 is available.


In [22]:
conversation_history = []

In [23]:
conversation_history.append({
    "role": "user",
    "content": "Can we have a meeting on Monday at 2 PM?"
})

conversation_history.append({
    "role": "assistant",
    "content": "Yes, Monday at 14:00 is available."
})

conversation_history

[{'role': 'user', 'content': 'Can we have a meeting on Monday at 2 PM?'},
 {'role': 'assistant', 'content': 'Yes, Monday at 14:00 is available.'}]

In [24]:
def chat_with_memory(user_message):
    # Add the user's message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Keep only the last 5 messages
    recent_history = conversation_history[-5:]

    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[
            {
                "role": "system",
                "content": """You are an AI meeting scheduler assistant.
Use the conversation history to understand the user's request.
Remember previous meeting-related information when relevant.
Answer clearly and briefly."""
            },
            *recent_history
        ],
        max_tokens=150
    )

    assistant_message = response.choices[0].message.content

    # Add assistant's response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message

In [25]:
chat_with_memory("What about Tuesday at 1 PM?")

'Tuesday at 13:00 is available.'

In [26]:
chat_with_memory("What about 3 PM instead?")

'Tuesday at 15:00 is available.'

In [34]:
def smart_scheduler(user_message):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Keep the last 5 messages for conversation memory
    recent_history = conversation_history[-5:]

    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[
            {
                "role": "system",
                "content": """You are an AI meeting scheduling assistant.

Use the conversation history to understand the user's request.

Your job is to identify the meeting day, exact time, or time period.

Rules:
- Day must be Monday, Tuesday, Wednesday, Thursday, or Friday.
- If the user gives an exact time, convert it to 24-hour HH:MM format.
- If the user says morning, afternoon, or evening, do not choose an exact time.
- For a time period, return the period instead.
- If the user gives only a day, return time: None and period: None.
- If the user does not provide a day, use the relevant day from previous conversation.

Return ONLY these three lines:

day: <day>
time: <HH:MM or None>
period: <morning, afternoon, evening, or None>

Examples:

User: Monday at 2 PM
day: Monday
time: 14:00
period: None

User: Monday afternoon
day: Monday
time: None
period: afternoon

User: Tuesday morning
day: Tuesday
time: None
period: morning

User: What times are available on Tuesday?
day: Tuesday
time: None
period: None"""
            },
            *recent_history
        ],
        max_tokens=80
    )

    llm_output = response.choices[0].message.content.strip()

    # Extract day, time, and period safely
    lines = llm_output.split("\n")

    day = "None"
    time = "None"
    period = "None"

    for line in lines:
        if line.lower().startswith("day:"):
            day = line.split(":", 1)[1].strip()

        elif line.lower().startswith("time:"):
            time = line.split(":", 1)[1].strip()

        elif line.lower().startswith("period:"):
            period = line.split(":", 1)[1].strip().lower()

    # If no valid day was found
    if day.lower() == "none":
        assistant_message = (
            "Please specify which day you would like to schedule the meeting."
        )

    # --------------------------------------------------
    # EXACT TIME REQUEST
    # --------------------------------------------------
    elif time.lower() != "none":

        matching_slots = df[
            (df["day"].str.lower() == day.lower()) &
            (df["time"] == time) &
            (df["status"] == "available")
        ]

        if len(matching_slots) > 0:

            assistant_message = (
                f"Yes, {day} at {time} is available."
            )

        else:

            alternatives = df[
                (df["day"].str.lower() == day.lower()) &
                (df["status"] == "available")
            ][["day", "time"]]

            if len(alternatives) > 0:

                alternative_times = ", ".join(
                    alternatives["time"].tolist()
                )

                assistant_message = (
                    f"Sorry, {day} at {time} is not available. "
                    f"Available times on {day} are: "
                    f"{alternative_times}."
                )

            else:

                assistant_message = (
                    f"Sorry, there are no available slots on {day}."
                )

    # --------------------------------------------------
    # DAY ONLY / NO TIME PERIOD
    # --------------------------------------------------
    elif period == "none":

        available_slots = df[
            (df["day"].str.lower() == day.lower()) &
            (df["status"] == "available")
        ][["day", "time"]]

        if len(available_slots) > 0:

            times = ", ".join(
                available_slots["time"].tolist()
            )

            assistant_message = (
                f"Available times on {day} are: {times}."
            )

        else:

            assistant_message = (
                f"Sorry, there are no available slots on {day}."
            )

    # --------------------------------------------------
    # TIME PERIOD REQUEST
    # --------------------------------------------------
    else:

        available_slots = df[
            (df["day"].str.lower() == day.lower()) &
            (df["status"] == "available")
        ].copy()

        if period == "morning":

            available_slots = available_slots[
                available_slots["time"].apply(
                    lambda x: int(x.split(":")[0]) < 12
                )
            ]

        elif period == "afternoon":

            available_slots = available_slots[
                available_slots["time"].apply(
                    lambda x: 12 <= int(x.split(":")[0]) < 17
                )
            ]

        elif period == "evening":

            available_slots = available_slots[
                available_slots["time"].apply(
                    lambda x: int(x.split(":")[0]) >= 17
                )
            ]

        if len(available_slots) > 0:

            times = ", ".join(
                available_slots["time"].tolist()
            )

            assistant_message = (
                f"Available {period} slots on {day} are: {times}."
            )

        else:

            assistant_message = (
                f"Sorry, there are no available {period} "
                f"slots on {day}."
            )

    # Save assistant response to conversation memory
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message

In [35]:
conversation_history = []

In [36]:
smart_scheduler("Can we have a meeting on Monday at 2 PM?")

'Yes, Monday at 14:00 is available.'

In [37]:
smart_scheduler("What times are available on Tuesday?")

'Available times on Tuesday are: 11:00, 13:00, 16:00.'

In [38]:
smart_scheduler("How about Tuesday at 4 PM?")

'Yes, Tuesday at 16:00 is available.'

In [ ]:
conversation_history = []

smart_scheduler("Can we have a meeting on Monday afternoon?")

In [ ]:
%pip install gradio

In [ ]:
import gradio as gr

print("Gradio is installed successfully!")

In [32]:
conversation_history = []

smart_scheduler("Can we have a meeting on Monday at 2 PM?")

LLM output:
day: Monday
time: 14:00
period: None


'Yes, Monday at 14:00 is available.'

In [33]:
smart_scheduler("What about Tuesday?")

LLM output:
day: Tuesday
time: None
period: None


'Available none slots on Tuesday are: 11:00, 13:00, 16:00.'

In [52]:
import gradio as gr

def chat_function(message, history):
    return smart_scheduler(message)

demo = gr.ChatInterface(
    fn=chat_function,
    title="🤖 SafeX AI Meeting Scheduler",
    description=(
        "Your AI-powered meeting scheduling assistant. "
        "Ask about available meeting times using natural language."
    ),
    textbox=gr.Textbox(
        placeholder="Example: Can we have a meeting on Monday at 2 PM?",
        label="Meeting Request",
        container=True
    )
)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [41]:
test_prompts = [
    "Can we have a meeting on Monday at 2 PM?",
    "Is Tuesday at 2 PM available?",
    "What times are available on Tuesday?",
    "Do you have anything available Wednesday morning?",
    "Can we meet Thursday afternoon?",
    "Is Friday at 2 PM available?",
    "What is available on Monday morning?",
    "Can we schedule something on Wednesday at 5 PM?",
    "What about Thursday?",
    "Can we meet on Friday at 9 AM?",
    "Is Monday at 10 AM free?",
    "Do you have anything available Tuesday afternoon?",
    "I need a meeting on Wednesday at 11 AM.",
    "Can you tell me a joke?",
    "What is the weather today?"
]

test_results = [
    [1, "Monday at 2 PM", "Check Monday 14:00 availability", "Passed", "Good", "None"],
    [2, "Tuesday at 2 PM", "Reject 14:00 and show alternatives", "Passed", "Good", "None"],
    [3, "What times are available on Tuesday?", "Show 11:00, 13:00, 16:00", "Passed", "Good", "None"],
    [4, "Wednesday morning", "Show Wednesday morning slot(s)", "Passed", "Good", "None"],
    [5, "Thursday afternoon", "Show Thursday afternoon slot(s)", "Passed", "Good", "None"],
    [6, "Friday at 2 PM", "Check Friday 14:00", "Passed", "Good", "None"],
    [7, "Monday morning", "Show Monday morning slot(s)", "Passed", "Good", "None"],
    [8, "Wednesday at 5 PM", "Check Wednesday 17:00", "Passed", "Good", "None"],
    [9, "What about Thursday?", "Show Thursday available slots", "Passed", "Good", "None"],
    [10, "Friday at 9 AM", "Check Friday 09:00", "Failed", "N/A", "Hugging Face 402 quota error"],
    [11, "Monday at 10 AM", "Check Monday 10:00", "Not executed", "N/A", "API quota exhausted"],
    [12, "Tuesday afternoon", "Show Tuesday afternoon slots", "Not executed", "N/A", "API quota exhausted"],
    [13, "Wednesday at 11 AM", "Check Wednesday 11:00", "Not executed", "N/A", "API quota exhausted"],
    [14, "Tell me a joke", "Politely refuse because it is out of scope", "Not executed", "N/A", "API quota exhausted"],
    [15, "What is the weather today?", "Politely refuse because it is out of scope", "Not executed", "N/A", "API quota exhausted"]
]

test_df = pd.DataFrame(
    test_results,
    columns=[
        "Test #",
        "User Prompt",
        "Expected Behavior",
        "Result",
        "Response Quality",
        "Failure / Notes"
    ]
)

test_df

In [47]:
test_df.to_csv("week2_test_results.csv", index=False)

print("Test sheet saved successfully.")

Test sheet saved successfully.


In [48]:
import os

print(os.path.abspath("week2_test_results.csv"))

D:\AI_Meeting_Scheduler_Assistant\Week_2\notebooks\week2_test_results.csv


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\dell\AppData\Local\Programs\Python\Python314\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\AppData\Local\Programs\Python\Python314\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


# SafeX AI Meeting Scheduler Assistant
## Client Pitch

### The Problem

Businesses spend significant time coordinating meetings through emails, messages, and manual calendar checks.

Employees often need to:

- Check multiple available time slots
- Respond to scheduling requests
- Handle follow-up questions
- Find alternative times when a requested slot is unavailable
- Manage repetitive scheduling conversations

This manual process creates unnecessary administrative work and delays.

### The SafeX AI Solution

SafeX can provide an AI-powered Meeting Scheduler Assistant that allows employees or customers to request meetings using normal conversational language.

For example:

> "Can we have a meeting on Tuesday afternoon?"

The AI understands the request, checks calendar availability, and responds with suitable meeting times.

If a requested time is unavailable, the assistant can immediately suggest alternatives.

The assistant can also remember recent conversation context, allowing users to ask natural follow-up questions such as:

> "What about Wednesday?"

### How It Helps the Business

The SafeX AI Meeting Scheduler can:

- Automate repetitive scheduling conversations
- Reduce manual calendar checking
- Provide faster responses
- Improve the customer and employee experience
- Reduce scheduling-related administrative work
- Operate through a simple conversational interface
- Be extended to connect with real business calendars

### Expected ROI

The main financial benefit comes from reducing the amount of employee time spent on repetitive scheduling tasks.

For example, if a business currently spends **10 hours per week** on scheduling-related administrative work and automation reduces that workload by **60%**, approximately **6 hours per week** could be redirected to higher-value activities.

That represents approximately:

**6 hours × 52 weeks = 312 hours saved per year.**

The actual ROI would depend on the organization's employee costs, meeting volume, and scheduling workload.

### Future Expansion

The prototype can be expanded to support:

- Google Calendar or Microsoft Outlook integration
- Real-time calendar synchronization
- Meeting duration
- Multiple participants
- Time zones
- Automatic meeting invitations
- Email notifications
- Business-specific scheduling rules

### Call to Action

SafeX can help businesses turn repetitive scheduling processes into intelligent AI-powered workflows.

**Let's identify one scheduling workflow in your business that can be automated and build a solution around it.**

# SafeX AI Meeting Scheduler Assistant
## How It Works

### 1. The user makes a request
The user types a normal meeting request, such as:

> "Can we have a meeting on Monday at 2 PM?"

The assistant understands natural language, so the user does not need to follow a strict format.

### 2. AI understands the request
The AI identifies important scheduling information such as:

- Meeting day
- Exact meeting time
- Time period such as morning, afternoon, or evening

For example:

**Monday at 2 PM → Monday, 14:00**

### 3. The assistant checks the calendar
SafeX's AI assistant checks a mock calendar containing available meeting slots.

For example, Tuesday has:

- 11:00 — Available
- 13:00 — Available
- 16:00 — Available

### 4. The assistant finds a suitable time
If the requested time is available, the assistant confirms it.

**Example:**

> "Yes, Monday at 14:00 is available."

If the requested time is unavailable, the assistant provides alternative available times.

**Example:**

> "Sorry, Tuesday at 14:00 is not available. Available times on Tuesday are: 11:00, 13:00, 16:00."

### 5. The assistant remembers the conversation
The assistant keeps the most recent conversation messages so users can ask natural follow-up questions.

For example:

**User:**  
"Is Monday at 2 PM available?"

**Assistant:**  
"Yes, Monday at 14:00 is available."

**User:**  
"What about Tuesday?"

The assistant can understand that the user is still discussing meeting availability.

### 6. Simple chat interface
The assistant is provided through a SafeX-branded chat interface.

Users can type multiple scheduling requests in one conversation instead of submitting each request separately.

### Benefits for a business

- Saves time spent checking calendars manually
- Makes scheduling easier for employees and customers
- Provides immediate responses
- Reduces back-and-forth communication
- Makes meeting availability easier to understand
- Can be extended to connect with real calendars in the future

### In simple terms

**User request → AI understands it → Calendar is checked → Best available options are returned**

The goal of SafeX's AI Meeting Scheduler Assistant is to make meeting coordination faster, simpler, and more automated.

# SafeX Cold Outreach Email

**Subject: Automating Meeting Scheduling with AI**

Hi,

I’m reaching out from SafeX to introduce our AI automation services.

Many businesses spend valuable employee time handling repetitive meeting scheduling requests, checking availability, and finding alternative times.

SafeX can automate this process with an AI Meeting Scheduler Assistant that understands natural-language requests, checks calendar availability, remembers conversation context, and suggests suitable meeting times.

For example, a user can simply ask:

"Can we meet Tuesday afternoon?"

The AI can identify available times and respond immediately.

This can help reduce administrative workload, speed up scheduling, and allow your team to focus on higher-value activities.

We would be happy to discuss one scheduling workflow in your organization that could benefit from AI automation.

Would you be available for a short conversation to explore the opportunity?

Best regards,  
Farid Ullah  
SafeX AI Automation Team  
faridniazi@gmail.com